# Auditoría de coherencia del PAA

Verifica el nuevo flujo de niveles de actividad FIRMS y la ausencia de conteo contemporáneo en los predictores.

In [1]:
from pathlib import Path
import pandas as pd
N=Path("data/processed/modelado/dataset_niveles_actividad_firms.parquet");M=Path("data/processed/modelado/dataset_modelado_nivel_actividad_firms.parquet")
n=pd.read_parquet(N);d=pd.read_parquet(M)
for x in (n,d):x["fecha_inicio_semana"]=pd.to_datetime(x.fecha_inicio_semana)
CLAVES=["departamento","fecha_inicio_semana"]

## 1. Integridad y target

In [2]:
for nombre,x in {"niveles":n,"modelado":d}.items():
 assert len(x)==7923 and x.departamento.nunique()==19 and not x.duplicated(CLAVES).any()
 assert x.fecha_inicio_semana.min()==pd.Timestamp("2018-01-01") and x.fecha_inicio_semana.max()==pd.Timestamp("2025-12-22")
assert set(n.nivel_actividad_firms)=={"Sin actividad","Bajo","Moderado","Alto"}
assert n.loc[n.cantidad_detecciones.eq(0),"nivel_actividad_firms"].eq("Sin actividad").all()
assert n.loc[n.cantidad_detecciones.gt(0),"nivel_actividad_firms"].ne("Sin actividad").all()
print("Integridad y target correctos")

Integridad y target correctos


## 2. Lags y leakage

In [3]:
LAGS=[f"cantidad_detecciones_lag{i}" for i in range(1,5)]
# Se comprueban desde la quinta fila objetivo; las primeras usan historia 2017 que no está en este panel.
o=d.sort_values(CLAVES)
for lag in range(1,5):
 esp=o.groupby("departamento").cantidad_detecciones.shift(lag);mask=esp.notna();assert o.loc[mask,LAGS[lag-1]].eq(esp[mask]).all()
X=["departamento","mes","temperature_2m_max_mean_weekly","relative_humidity_2m_min_mean_weekly","wind_speed_10m_max_mean_weekly","precipitation_sum_weekly"]+LAGS+["promedio_detecciones_ultimas_4_semanas","max_detecciones_ultimas_4_semanas","semanas_con_actividad_ultimas_4"]
assert {"cantidad_detecciones","nivel_actividad_firms","nivel_actividad_firms_codigo"}.isdisjoint(X) and not d[X].isna().any().any();print("Lags y exclusiones correctos")

Lags y exclusiones correctos


## 3. Estado

In [4]:
pd.DataFrame([
    ["Calidad", "Vigente", "Unidad departamento-semana, cobertura, ceros y asimetría"],
    ["Target", "Vigente", "Ceros separados y K-Means k=3 sobre log1p de conteos positivos"],
    ["Variables", "Vigente", "V1–V4; el conteo contemporáneo está excluido"],
    ["Modelos", "Vigente", "Dummy, Logística, Random Forest e HistGradientBoosting"],
    ["Selección", "Vigente", "Random Forest V1 seleccionado sólo con validation"],
    ["Evaluación", "Vigente", "Split estratificado 70/15/15; test usado tras fijar la solución"],
    ["Nesterov", "Complementario", "Fuera del target y del modelo final"],
], columns=["Componente", "Estado", "Evidencia"])

,Componente,Estado,Evidencia
0,Calidad,Vigente,"Unidad departamento-semana, cobertura, ceros y..."
1,Target,Vigente,Ceros separados y K-Means k=3 sobre log1p de c...
2,Variables,Vigente,V1–V4; el conteo contemporáneo está excluido
3,Modelos,Vigente,"Dummy, Logística, Random Forest e HistGradient..."
4,Selección,Vigente,Random Forest V1 seleccionado sólo con validation
5,Evaluación,Vigente,Split estratificado 70/15/15; test usado tras ...
6,Nesterov,Complementario,Fuera del target y del modelo final


## Conclusión

Flujo vigente: FIRMS histórico → log1p → K-Means → niveles de actividad → clasificación con meteorología, contexto y antecedentes. Actividad FIRMS no equivale a riesgo ni incendio confirmado.

## 4. Entregable 3: particiones comunes

In [5]:
import hashlib, json, joblib, sys
SPLITS=Path("data/processed/modelado/splits")
particiones={x:pd.read_parquet(SPLITS/f"{x}_keys.parquet") for x in ["train","validation","test"]}
assert {x:len(v) for x,v in particiones.items()}=={"train":5546,"validation":1188,"test":1189}
claves_particion={x:set(map(tuple,v[CLAVES].astype(str).to_numpy())) for x,v in particiones.items()}
assert claves_particion["train"].isdisjoint(claves_particion["validation"])
assert claves_particion["train"].isdisjoint(claves_particion["test"])
assert claves_particion["validation"].isdisjoint(claves_particion["test"])
assert set.union(*claves_particion.values())==set(map(tuple,d[CLAVES].astype(str).to_numpy()))
print("Particiones completas y sin solapamiento")

Particiones completas y sin solapamiento


## 5. Reserva de test y variables prohibidas

In [6]:
experimentales=[Path("experimentos")/f"Experimento_0{i}_{nombre}.ipynb" for i,nombre in [(1,"Dummy"),(2,"LogisticRegression"),(3,"RandomForest"),(4,"HistGradientBoosting")]]
for ruta in experimentales:
    nb=json.loads(ruta.read_text(encoding="utf-8"))
    codigo="\n".join("".join(c.get("source",[])) for c in nb["cells"] if c.get("cell_type")=="code")
    assert 'subset("test")' not in codigo and "test_keys.parquet" not in codigo, ruta
    assert 'PROHIBITED.isdisjoint' in codigo, ruta
print("Los notebooks experimentales no cargan test y controlan variables prohibidas")

Los notebooks experimentales no cargan test y controlan variables prohibidas


## 6. Resultados, artefacto e inferencia

In [7]:
RESULTADOS=Path("results/experimentos_entregable3.csv")
META=Path("artifacts/metadata_modelo.json");MODELO=Path("artifacts/modelo_final.joblib")
assert RESULTADOS.exists() and len(pd.read_csv(RESULTADOS))>0
assert META.exists() and MODELO.exists()
meta=json.loads(META.read_text(encoding="utf-8"));modelo=joblib.load(MODELO)
assert meta["target"]=="nivel_actividad_firms" and list(modelo.classes_)==sorted(meta["clases"])
assert "cantidad_detecciones" not in meta["variables"]
sys.path.insert(0,str(Path.cwd()))
from src.inference import predict
ejemplo=d[meta["variables"]].iloc[[0]]
salida=predict(ejemplo,modelo,meta)
assert salida[0] in meta["clases"]
print("Resultados, metadata, Pipeline e inferencia coherentes:",salida[0])

Resultados, metadata, Pipeline e inferencia coherentes: Moderado


## Conclusión del Entregable 3

Los splits son comunes y no se solapan; los notebooks individuales no acceden a test; las variables prohibidas no forman parte de X; y el artefacto serializado puede realizar inferencia sin reentrenamiento.